<a href="https://colab.research.google.com/github/Shahana023/cse-resources/blob/main/autonomous_vit/AV_adversarial_defense_proposal_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Adversarial Robustness of Transformer-Based Object Detectors for Autonomous-Vehicle Perception
**Area:** adversarial machine learning · autonomous-vehicle perception · AI security

An empirical study on a single KITTI driving frame. All figures and quantities are produced by the code below.

- **Part A.** An $\ell_\infty$-bounded, near-imperceptible perturbation that collapses DETR's detection set.
- **Part B.** A randomized-smoothing detection statistic that identifies and localizes such a perturbation.
- **Part C.** The statistic under a defence-aware (adaptive) adversary, its inference cost, and the resulting open problems.

## Step 0 — Setup

In [ ]:
!pip install -q transformers timm torch torchvision pillow matplotlib requests hf_transfer

### (Optional) Hugging Face token
The model is public, so no token is required; it only speeds the download. Paste one into `HF_TOKEN = ""` below, or leave it blank. On Kaggle a token can instead be added under **Add-ons ▸ Secrets** as `HF_TOKEN`.

In [ ]:
import os
HF_TOKEN = ""   # optional
_tok = HF_TOKEN.strip()
if not _tok:
    try:
        from google.colab import userdata; _tok = userdata.get("HF_TOKEN") or ""
    except Exception: pass
if not _tok:
    try:
        from kaggle_secrets import UserSecretsClient; _tok = UserSecretsClient().get_secret("HF_TOKEN") or ""
    except Exception: pass
if _tok: os.environ["HF_TOKEN"] = _tok; print("HF token set.")
else: print("No token set; continuing with the public model.")

## Step 1 — Load a pretrained DETR detector

In [ ]:
import os, warnings; warnings.filterwarnings("ignore")
import time, numpy as np, torch, torch.nn.functional as F, requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from transformers import DetrImageProcessor, DetrForObjectDetection
from transformers.utils import logging as _hflog; _hflog.set_verbosity_error()
try:
    import hf_transfer; os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
except Exception: pass
torch.manual_seed(0); np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
IMG = (320, 1024)                                  # (H, W)
proc = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50").to(device).eval()
for p in model.parameters(): p.requires_grad_(False)
MEAN = torch.tensor(proc.image_mean, device=device).view(1,3,1,1)
STD  = torch.tensor(proc.image_std,  device=device).view(1,3,1,1)
NO = model.config.num_labels; COCO = model.config.id2label
norm = lambda x: (x - MEAN) / STD

def detect(x, th=0.7):
    with torch.no_grad(): out = model(pixel_values=norm(x))
    return proc.post_process_object_detection(out, target_sizes=torch.tensor([IMG], device=device), threshold=th)[0]
def n_obj(x): return len(detect(x)["scores"])
def fg(x): return 1 - model(pixel_values=norm(x)).logits.softmax(-1)[..., NO]   # P(object) per query
def show(x, res, title, ax):
    ax.imshow(np.clip(x.squeeze(0).permute(1,2,0).detach().cpu().numpy(), 0, 1))
    for s,l,b in zip(res["scores"], res["labels"], res["boxes"]):
        x0,y0,x1,y1 = b.tolist()
        ax.add_patch(mpatches.Rectangle((x0,y0),x1-x0,y1-y0,lw=2,edgecolor="lime",facecolor="none"))
        ax.text(x0,y0-4,f'{COCO[l.item()]}:{s:.2f}',color="lime",fontsize=7,bbox=dict(facecolor="black",alpha=0.5,pad=1))
    ax.set_title(title); ax.axis("off")
print("DETR loaded on", device)

## Step 2 — Input image (KITTI)

In [ ]:
r = requests.get("https://raw.githubusercontent.com/open-mmlab/mmdetection3d/main/demo/data/kitti/000008.png",
                 timeout=30, headers={"User-Agent": "Mozilla/5.0"})
img = torch.from_numpy(np.array(Image.open(BytesIO(r.content)).convert("RGB").resize((IMG[1], IMG[0]))
                                 ).astype(np.float32)/255).permute(2,0,1).unsqueeze(0).to(device)
print("image:", tuple(img.shape))

# Part A — A bounded adversarial perturbation

### Step 3 — Baseline detection

In [ ]:
base = detect(img); n0 = len(base["scores"])
fig, ax = plt.subplots(figsize=(14,4)); show(img, base, f"DETR: {n0} objects detected", ax); plt.show()
print("baseline detections:", n0)

### Step 4 — Projected-gradient perturbation
A projected-gradient (PGD) perturbation (Madry et al., 2018) constrained to an $\ell_\infty$ ball of radius $\varepsilon = 8/255 \approx 0.03$, minimising the mean top-$k$ foreground confidence of the detector. This is a **white-box** threat model (the attack has access to the detector's gradients); a 2026 benchmark (arXiv:2602.16494) reports that attacks *transferred* from CNN detectors are markedly less effective on transformer detectors, so the vulnerability shown here is specifically the white-box — and, in Part C, defence-aware — setting.

In [ ]:
EPS, STEPS, ALPHA = 8/255, 30, 2/255
def pgd(steps=STEPS, eps=EPS, adaptive=False, M=4, sigma=0.10):
    a = img.clone()
    for _ in range(steps):
        a.requires_grad_(True)
        if adaptive:                                   # optimise under the smoothing noise (EOT)
            loss = sum(fg((a+sigma*torch.randn_like(a)).clamp(0,1)).topk(min(20,fg(a).shape[1]),dim=1).values.mean() for _ in range(M))/M
        else:
            loss = fg(a).topk(min(20, fg(a).shape[1]), dim=1).values.mean()
        g = torch.autograd.grad(loss, a)[0]
        a = torch.min(torch.max((a - ALPHA*g.sign()).detach(), img-eps), img+eps).clamp(0,1)
    return a

adv = pgd()
atk = detect(adv); nA = len(atk["scores"]); linf = (adv-img).abs().max().item()
fig, ax = plt.subplots(1,2,figsize=(20,5))
show(img, base, f"Clean: {n0} objects", ax[0])
show(adv, atk, f"Perturbed: {nA} objects", ax[1]); plt.tight_layout(); plt.show()
print(f"detections: {n0} -> {nA}   (L_inf perturbation = {linf:.3f})")

### Step 5 — Magnitude of the perturbation
The perturbation, contrast-amplified for visibility. Its true per-pixel magnitude is $\approx 0.03$, below the threshold of human perception.

In [ ]:
pert = (adv - img).squeeze(0).permute(1,2,0).detach().cpu().numpy()
fig, ax = plt.subplots(figsize=(14,4))
ax.imshow(np.clip((pert - pert.min())/(pert.max() - pert.min() + 1e-9), 0, 1)); ax.axis("off")
ax.set_title(f"Perturbation (contrast-amplified). True per-pixel magnitude: {linf:.3f}.")
plt.show()

# Part B — A randomized-smoothing detection signal

Define the smoothed detector by averaging over noise. With $N(x)$ the number of detections and $\varepsilon\sim\mathcal N(0,\sigma^2 I)$,
$$I(x)=\mathbb E_{\varepsilon}\big[N(x+\varepsilon)\big]-N(x).$$
For a clean image, noise removes only weak detections, so $I(x)\le 0$. For an image whose detections were suppressed by a small perturbation, the perturbation is fragile to noise and the detections reappear, so $I(x)>0$. This follows the randomized-smoothing framework of Cohen et al. (2019); the regions where detections reappear indicate where objects were suppressed.

In [ ]:
def instability(x, sigma, K=20):
    nb = n_obj(x)
    ns = np.mean([n_obj((x + sigma*torch.randn_like(x)).clamp(0,1)) for _ in range(K)])
    return nb, ns, ns - nb

print(f"{'sigma':>6}{'clean I(x)':>12}{'perturbed I(x)':>16}")
for sigma in [0.05, 0.10, 0.15]:
    _,_,Ic = instability(img, sigma); _,_,Ia = instability(adv, sigma)
    print(f"{sigma:>6}{Ic:>+12.1f}{Ia:>+16.1f}")

### Localization of the suppressed objects

In [ ]:
sigma = 0.10; K = 24
heat = np.zeros(IMG)
for _ in range(K):
    for b in detect((adv + sigma*torch.randn_like(adv)).clamp(0,1))["boxes"]:
        x0,y0,x1,y1 = [int(v) for v in b.tolist()]
        heat[max(0,y0):min(IMG[0],y1), max(0,x0):min(IMG[1],x1)] += 1
fig, ax = plt.subplots(figsize=(14,4))
ax.imshow(np.clip(adv.squeeze(0).permute(1,2,0).cpu().numpy(),0,1)); ax.imshow(heat, cmap="jet", alpha=0.45)
ax.set_title("Perturbed frame; regions where detections reappear under noise"); ax.axis("off")
plt.show()

# Part C — A defence-aware attacker, runtime cost, and open problems

The signal in Part B was evaluated against an attacker with no knowledge of it. The following evaluates it against an attacker that optimises the perturbation to keep the detections suppressed *under the smoothing noise* (Expectation over Transformation).

### Step 6 — Attacker with knowledge of the defence

In [ ]:
_,_,I_na = instability(adv, 0.10)                             # attacker unaware of the defence
adv_ad = pgd(steps=50, eps=EPS, adaptive=True, M=4, sigma=0.10)  # attacker optimising under noise
_,_,I_ad = instability(adv_ad, 0.10)
print(f"no-knowledge attack :  I(x) = {I_na:+.1f}   (L_inf = {(adv-img).abs().max().item():.3f})")
print(f"defence-aware attack:  I(x) = {I_ad:+.1f}   (L_inf = {(adv_ad-img).abs().max().item():.3f})")

At the same perturbation budget, the defence-aware attacker returns $I(x)$ to its clean-image range. Detection-based defences are known to be vulnerable to such adaptive attacks (Athalye et al., 2018); robustness against a defence-aware adversary is therefore established through provable guarantees rather than through evaluation against fixed attacks.

### Step 7 — Runtime cost

In [ ]:
K = 24
t0 = time.time()
for _ in range(K): _ = n_obj((adv + 0.10*torch.randn_like(adv)).clamp(0,1))
dt = time.time() - t0; fps = 1/dt
print(f"cost: {K} forward passes = {dt*1000:.0f} ms/frame  ->  {fps:.2f} FPS")
print(f"real-time target ~30 FPS  ->  requires ~{30/fps:.0f}x speedup.")

The signal requires $K$ forward passes per frame. Meeting a real-time budget (~30 FPS) would require reducing this cost — for example, a tiered scheme that runs an inexpensive check on every frame and invokes the smoothing computation selectively.

### Open problems
- Certified robustness to adversarial patches has been established for CNN image classifiers but not for transformer detectors such as DETR, whose global attention does not admit the receptive-field arguments those certificates rely on.
- The perturbation here is digital; physically realizable attacks (e.g. printed patches) are a distinct and harder setting.
- The evaluation is a single frame; dataset-scale metrics (mAP, attack-success rate over KITTI / BDD100K) are required to characterize the method.

### Summary
On a single KITTI frame, a bounded perturbation suppressed all detections (Part A); a randomized-smoothing signal detected and localized it (Part B); a defence-aware attacker evaded the signal at the same perturbation budget, the signal's runtime exceeds a real-time budget, and certified guarantees for transformer detectors, physical realizability, and dataset-scale evaluation remain open (Part C).

## References

*Surveys — the field map (2025):*
- *A Survey of Adversarial Defenses in Vision-based Systems.* 2025. (arXiv:2503.00384)
- *Revisiting Adversarial Perception Attacks and Defense Methods on Autonomous Driving Systems.* 2025. (arXiv:2505.11532)

*State of the art and related defences (2024–2026):*
- *Revisiting Adversarial Patch Defenses on Object Detectors* (adaptive-evaluation benchmark). ICCV 2025. (arXiv:2508.00649)
- NutNet. ACM CCS 2024. (arXiv:2406.10285) &middot; PhySense. ACM CCS 2024. &middot; Physics-Aware Spatiotemporal Consistency. Sensors 2026.
- AdvPatchXAI. CVPR 2025. &middot; X-Detect. Machine Learning (Springer) 2024. (arXiv:2306.08422) &middot; Explainable AI-guided Test-Time Defense for YOLO. FGCS 2026.
- Nazeri, Zhao & Pisu. *Evaluating the Adversarial Robustness of Detection Transformers.* Applied Intelligence 2026. (arXiv:2412.18718)
- *Benchmarking Adversarial Robustness for Object Detection.* 2026. (arXiv:2602.16494) &middot; LipSSD. 2026. (arXiv:2607.06592) &middot; *A Single Set of Adversarial Clothes Breaks Multiple Defenses.* 2025. (arXiv:2510.17322)

*Foundational techniques applied in the demo (standard methods, cited to origin):* projected-gradient attack — Madry et al., ICLR 2018; Expectation-over-Transformation — Athalye et al., ICML 2018; adaptive-attack evaluation — Athalye, Carlini & Wagner, ICML 2018; randomized smoothing — Cohen, Rosenfeld & Kolter, ICML 2019.